# Custom Good/Bad Trainer

Upload your own images for any item, sort them into `good` / `bad`, and train a MobileNetV2 classifier — same pipeline as `component_classifier_fixed.ipynb`, just with an upload step instead of a fixed dataset folder.

Requires `ipywidgets>=8` (`pip install ipywidgets`). Set `item_name` below before running.

In [ ]:
import os
from pathlib import Path
import ipywidgets as widgets
from IPython.display import display

In [ ]:
item_name = "my_item"  # change this for each new item you want to train

good_dir = Path(f"custom_dataset/{item_name}/good")
bad_dir = Path(f"custom_dataset/{item_name}/bad")
good_dir.mkdir(parents=True, exist_ok=True)
bad_dir.mkdir(parents=True, exist_ok=True)

`item_name` controls where images and the trained model are saved, so you can train several different items without overwriting each other.

In [ ]:
good_upload = widgets.FileUpload(accept="image/*", multiple=True, description="Good images")
bad_upload = widgets.FileUpload(accept="image/*", multiple=True, description="Bad images")
save_button = widgets.Button(description="Save uploaded images")
save_output = widgets.Output()

def on_save_clicked(b):
    with save_output:
        for file_info in good_upload.value:
            (good_dir / file_info["name"]).write_bytes(file_info["content"])
        for file_info in bad_upload.value:
            (bad_dir / file_info["name"]).write_bytes(file_info["content"])
        print(f"Saved {len(good_upload.value)} good and {len(bad_upload.value)} bad images to custom_dataset/{item_name}")

save_button.on_click(on_save_clicked)
display(good_upload, bad_upload, save_button, save_output)

Upload images with the two buttons above, then click **Save uploaded images**. Repeat as many times as you like — new uploads add on top of what's already saved. You can also skip this widget entirely and drop images straight into `custom_dataset/<item_name>/good` and `/bad` yourself.

In [ ]:
print("good:", len(list(good_dir.glob("*"))))
print("bad:", len(list(bad_dir.glob("*"))))

Sanity check before training — make sure both folders actually have images in them, and that the counts aren't wildly imbalanced (a handful of images per class won't train a usable model).

In [ ]:
import tensorflow as tf
print("TF version:", tf.__version__)
print("GPU available:", tf.config.list_physical_devices("GPU"))

In [ ]:
data_dir = Path(f"custom_dataset/{item_name}")

img_size = (224, 224)
batch_size = 32
seed = 42

train_ds = tf.keras.utils.image_dataset_from_directory(
    data_dir,
    validation_split=0.2,
    subset="training",
    seed=seed,
    image_size=img_size,
    batch_size=batch_size
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    data_dir,
    validation_split=0.2,
    subset="validation",
    seed=seed,
    image_size=img_size,
    batch_size=batch_size
)

class_names = train_ds.class_names
print(class_names)

Same 224x224 input size as the existing classifier, since it feeds the same MobileNetV2 base. Labels come from folder names, so `bad` = 0, `good` = 1.

In [ ]:
import numpy as np

labels = np.concatenate([y.numpy() for _, y in train_ds])
counts = np.bincount(labels)
print("bad:", counts[0], "good:", counts[1])

total = counts.sum()
class_weight = {
    0: total / (2 * counts[0]),
    1: total / (2 * counts[1])
}
print("class weights:", class_weight)

Uploaded sets are rarely balanced, so class weights are computed here and passed into training, same as the original notebook.

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal_and_vertical"),
    tf.keras.layers.RandomRotation(0.15),
    tf.keras.layers.RandomZoom(0.15),
    tf.keras.layers.RandomContrast(0.1),
])

train_ds = train_ds.map(lambda x, y: (data_augmentation(x, training=True), y), num_parallel_calls=AUTOTUNE)

train_ds = train_ds.cache().shuffle(1000).prefetch(AUTOTUNE)
val_ds = val_ds.cache().prefetch(AUTOTUNE)

Augmentation matters even more here since uploaded sets are usually small — it reduces overfitting to the exact photos you took.

In [ ]:
from tensorflow.keras import layers, models

base_model = tf.keras.applications.MobileNetV2(
    input_shape=img_size + (3,),
    include_top=False,
    weights="imagenet"
)
base_model.trainable = False  # freeze for the first training phase

preprocess_input = tf.keras.applications.mobilenet_v2.preprocess_input

inputs = tf.keras.Input(shape=img_size + (3,))
x = preprocess_input(inputs)
x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)
x = layers.Dense(128, activation="relu")(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(1, activation="sigmoid")(x)

model = tf.keras.Model(inputs, outputs)
model.summary()

Same transfer-learning setup as `component_classifier_fixed.ipynb`: frozen MobileNetV2 backbone plus a small classification head, so it works well even with a small uploaded dataset.

In [ ]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2, min_lr=1e-6)
]

In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=25,
    class_weight=class_weight,
    callbacks=callbacks
)

Phase 1: train the classification head with the backbone frozen. EarlyStopping cuts this short once validation loss stops improving.

In [ ]:
base_model.trainable = True

fine_tune_at = len(base_model.layers) - 30
for layer in base_model.layers[:fine_tune_at]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

fine_tune_history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=15,
    class_weight=class_weight,
    callbacks=callbacks
)

Phase 2: unfreeze the last 30 layers and fine-tune at a much lower learning rate so the model adapts to your specific item.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

y_true = np.concatenate([y.numpy() for _, y in val_ds])
y_pred_probs = model.predict(val_ds).ravel()
y_pred = (y_pred_probs > 0.5).astype(int)

print(classification_report(y_true, y_pred, target_names=class_names))
print("Confusion matrix:\n", confusion_matrix(y_true, y_pred))

Precision/recall/F1 per class plus the confusion matrix — worth checking before trusting the model, especially since uploaded datasets are often small and imbalanced.

In [ ]:
Path("models").mkdir(exist_ok=True)
model.save(f"models/{item_name}_classifier.keras")
print(f"Saved model to models/{item_name}_classifier.keras")

Saved as `models/<item_name>_classifier.keras` so it doesn't overwrite the existing `component_classifier.keras`. To use it in the inference service, either rename it to `component_classifier.keras` in the `models/` folder, or point the service at it directly by setting the `MODEL_PATH` env var (in `k8s/03-inference.yaml` or `docker-compose.yml`) to `/app/models/<item_name>_classifier.keras`.